In [1]:
# В данных были дубли

#df = pd.read_csv("C:/users/savateev_a_v/Nextcloud/Desktop/Макрос/Справочник РЦ.csv", sep=';')
#print(df.shape)
#dim_rc_clean = df.drop_duplicates()
#print(dim_rc_clean.shape)
#dim_rc_clean.to_parquet("C:/users/savateev_a_v/Nextcloud/Desktop/Макрос/dim_rc.parquet", index=False)

## Задачи

- Составить нормальную архетектуру файла
- При составление архетектуры учесть то что в данных может не быть (данных/столбца/неверное название файла)
- Добавить логирование
- Из цикла for можно сдеалать одну функцию, если передавать ей нужные столбы как аргумент
- Так же можно придумать еще две функции обработки данных для первого и второго случая, а также отдельно фунцию объединения, что б main просто запускался 4 функциями.
- Положить запрос sql в файл - sql
- Сделать запрос безопасным, положить пароли в .env

In [ ]:
import clickhouse_connect
import pandas as pd
import pyarrow
import openpyxl
import datetime as dt
from datetime import datetime
from pathlib import Path

from config import settings
import constants as const
from processing import attach_dim_rc
from io_utils import read_excel_snapshot
import db 

start=datetime.now()

# справочник регионов
dim_rc = pd.read_parquet(settings.DIM_RC_PATH)


In [ ]:
dim_district

In [53]:
dim_rc

,Названия строк,Cтандартные наименования РЦ,Федеральный округ,Код РЦ
0,АС,РЦ Астрахань Тинаки,Южный,300000
1,Астрахань,РЦ Астрахань Тинаки,Южный,300000
2,Астрахань РЦ,РЦ Астрахань Тинаки,Южный,300000
3,Астрахань Тинаки,РЦ Астрахань Тинаки,Южный,300000
4,Батайск,РЦ Батайск,Южный,611000
...,...,...,...,...
269,РЦ Первоуральск,РЦ Первоуральск,Уральский,660000
270,РЦ ОН,РЦ Оренбург Ленина,Приволжский,561000
271,РЦ КЛ,РЦ Коломна,Центральный,772000
272,Кропоткин,РЦ Кропоткин,Южный,231000


In [54]:
data_sg_mo = read_excel_snapshot(settings.TESE, const.MO_REQUIRED_COLUMNS)

c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\

In [55]:
data_sg_mo = attach_dim_rc(data_sg_mo, dim_rc)

In [58]:
data_sg_mo[data_sg_mo['Годен до'].isna()]

,Код ТП,Наименование ТП,Годен до,Количество по СГ,source_file,Названия строк,Cтандартные наименования РЦ,Федеральный округ,Код РЦ
1343,1000382819,ТОМАТЫ Flamenco сливовидные 450г:8,NaN,24.0,РЦ Астрахань Тинаки,РЦ Астрахань Тинаки,РЦ Астрахань Тинаки,Южный,300000
2300,1000475101,"Водка ПЧЕЛКА Люкс 38% 0,25л (Нама):12",NaN,12.0,РЦ Астрахань Тинаки,РЦ Астрахань Тинаки,РЦ Астрахань Тинаки,Южный,300000
2318,1000499757,"Водка БЕЛАЯ БЕРЕЗКА 40% 0,7л (Омсквинпром):12",NaN,132.0,РЦ Астрахань Тинаки,РЦ Астрахань Тинаки,РЦ Астрахань Тинаки,Южный,300000
2322,1000275537,"Водка ХАСКИ 40% 0,7л(Россия):12",NaN,264.0,РЦ Астрахань Тинаки,РЦ Астрахань Тинаки,РЦ Астрахань Тинаки,Южный,300000
2334,1000282726,Вино ВЕДЕРНИКОВЪ Донское Сибирьковый бел п/сух...,NaN,48.0,РЦ Астрахань Тинаки,РЦ Астрахань Тинаки,РЦ Астрахань Тинаки,Южный,300000
...,...,...,...,...,...,...,...,...,...
240344,8000013176,"Карта лояльности ""Магнит плюс"":100/1000",NaN,2900.0,РЦ Ярославль,РЦ Ярославль,РЦ Ярославль,Центральный,760000
240478,2149400013,"Ясно солнышко Хлопья гречневые 0,38кг (ПМК):9",NaN,9.0,РЦ Ярославль,РЦ Ярославль,РЦ Ярославль,Центральный,760000
240592,1000489375,ЯБЛОКОВ Чипсы яблочные кисло-сладкие 25г фл/п(...,NaN,32.0,РЦ Ярославль,РЦ Ярославль,РЦ Ярославль,Центральный,760000
240802,3441610038,ЛИСМА Чай крепк. 100пак./2г :6,NaN,6.0,РЦ Ярославль,РЦ Ярославль,РЦ Ярославль,Центральный,760000


In [16]:
#data_sg_mo[data_sg_mo['Годен до'].isna()]
data_sg_mo['source_file'].value_counts()
data_sg_mo[data_sg_mo['source_file'].str.contains('Ногин')]

,Код ТП,Наименование ТП,Годен до,Количество по СГ,source_file
113576,3367460002,(ПФ) СОЛЬ Поваренная пищевая 1кг,29.11.2027 00:00:00,750.00,РЦ Ногинск (а)
113577,3367460006,"Соль ""ЭКСТРА"" пищевая 1кг",11.10.2027 00:00:00,180.00,РЦ Ногинск (а)
113578,3367460006,"Соль ""ЭКСТРА"" пищевая 1кг",20.11.2027 00:00:00,40.00,РЦ Ногинск (а)
113579,9072651210,КАРТОФЕЛЬ 1кг,27.01.2026 00:00:00,4725.62,РЦ Ногинск (а)
113580,9072651210,КАРТОФЕЛЬ 1кг,28.01.2026 00:00:00,3274.20,РЦ Ногинск (а)
...,...,...,...,...,...
118743,1000563964,"TRUFFLE MILK Конфеты трюфель молочный 0,2кг(Эс...",24.12.2026 00:00:00,180.00,РЦ Ногинск (а)
118744,1000195954,ОТ МАРТИНА Семечки белые подсолнечн 100г фл/п(...,12.08.2026 00:00:00,1080.00,РЦ Ногинск (а)
118745,1000195954,ОТ МАРТИНА Семечки белые подсолнечн 100г фл/п(...,11.11.2026 00:00:00,140.00,РЦ Ногинск (а)
118746,1000169018,KOTANYI Лавровый лист 5г сашет(Котани):25,02.12.2029 00:00:00,1075.00,РЦ Ногинск (а)


Подключение и получение данных исключений из справочника

In [5]:
client = db.get_clickhouse_client()

# Справочник исключений и справочник окургов
dim_goods = db.get_excluded_goods(client)
dim_district = db.get_dim_district(client)

In [6]:
folder_mo = settings.MO_ROOT

folder_mo = max(  # Выбирает файл с поледней датой изменения
    [p for p in folder_mo.iterdir() if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)
#print(folder_mo.exists()) - чек статуса
excel_files_mo = list(folder_mo.glob("*.xlsx"))


folder_mx = settings.MX_ROOT
folder_mx = max(  # Выбирает файл с поледней датой изменения
    [p for p in folder_mx.iterdir() if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)

excel_files_mx = list(folder_mx.glob("*.xlsx"))

In [7]:
dfs = []

for file_path in excel_files_mo:
    df = pd.read_excel(file_path, usecols=const.MO_REQUIRED_COLUMNS)
    df["source_file"] = file_path.stem
    dfs.append(df)

data_sg_mo = pd.concat(dfs, ignore_index=True)

data_sg_mo = data_sg_mo.merge(
    dim_rc,
    how="left",
    left_on="source_file",
    right_on="Названия строк"
)

data_sg_mo = data_sg_mo.drop(columns=const.RC_MERGE_DROP_COLUMNS)

c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\

In [6]:
data_sg_mo["Годен до"] = pd.to_datetime(
    data_sg_mo["Годен до"],
    format=const.MO_EXPIRY_DATE_FORMAT
).dt.normalize()

In [7]:
data_sg_mo = data_sg_mo.groupby(
    const.MO_GROUPBY_COLUMNS,
    as_index=False
).agg(const.MO_AGG_MAP)

In [8]:
data_sg_mo["Код ТП"] = data_sg_mo["Код ТП"].astype(str)
dim_goods["Код"] = dim_goods["CODE"].astype(str)

In [9]:
# Удаляем данные из словаря товаров
print(data_sg_mo.shape)
data_sg_mo = data_sg_mo[~data_sg_mo['Код ТП'].isin(dim_goods["CODE"])]
print(data_sg_mo.shape)
data_sg_mo = data_sg_mo[~data_sg_mo["Наименование ТП"].str.contains("МРЦ", na=False)]
print(data_sg_mo.shape)

(243778, 5)
(207487, 5)
(207487, 5)


In [10]:
# Переименовал столбцы согласно макросу

data_sg_mo.columns = const.MO_FINAL_COLUMNS

In [11]:
dfs = []

for file_path in excel_files_mx:
    df = pd.read_excel(file_path, usecols=const.MX_REQUIRED_COLUMNS)
    df["source_file"] = file_path.stem
    dfs.append(df)

data_mx = pd.concat(dfs, ignore_index=True)

data_mx = data_mx.merge(
    dim_rc,
    how="left",
    left_on="source_file",
    right_on="Названия строк"
)

c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\savateev_a_v\dev\macros\venv\Lib\site-packages\

In [12]:
# Меняем тип данных для группировки и по дате и дальнейшего merge по коду

data_mx["Годен до"] = pd.to_datetime(
    data_mx["Годен до"],
    format=const.MX_EXPIRY_DATE_FORMAT
).dt.normalize()
data_mx["код"] = data_mx["код"].astype(str)

In [13]:
print(data_mx.shape) # Отсавляем только активные паллеты
data_mx = data_mx[~data_mx['Годен до'].isna() & data_mx['актив']==1]
print(data_mx.shape) # Убираем те товары которые начинаются с (РСБ, СВХ ОПТ)
data_mx = data_mx[~data_mx['Примечание'].str.startswith(("РСБ", "СВХ", "ОПТ"), na=False)]
print(data_mx.shape) # Убираем все ячейки которые срок годности = 5000
data_mx = data_mx[
    (data_mx['Срок хранения в КТ'] < 5000) |
    (data_mx['Срок хранения в КТ'].isna())
]
print(data_mx.shape) # Убираем все ячейки которые начинаются на "Корзина"
data_mx = data_mx[~data_mx['МП'].str.startswith('Корзина', na=False)]
print(data_mx.shape) # Убираем все где Годен до больше чем 5000 (так как это не возможно) и там где уже просрочка
now = dt.datetime.now()
data_mx= data_mx[(data_mx['Годен до'] < (now + dt.timedelta(days=5000))) & (data_mx['Годен до'] > (now - dt.timedelta(days=1)))]
print(data_mx.shape)


# Удаляем не нужные колонки

data_mx = data_mx.drop(columns=const.MX_DROP_COLUMNS_AFTER_FILTER)

(1023828, 12)
(698692, 12)
(698543, 12)
(560564, 12)
(560336, 12)
(559121, 12)


In [14]:
data_mx = data_mx.groupby(
    const.MX_GROUPBY_COLUMNS,
    as_index=False
).agg(const.MX_AGG_MAP)

In [15]:
# Работаем тут
# Получаю минимальную дату для МХ.
# Для того что свисти к минимому ошибку при поиске ошибок попнения
# Получаестя что мы берем минимально низкую дату харенеия для того что бы отсечь все спорные даты на МО

data_mx = data_mx.loc[
    data_mx.groupby(['Cтандартные наименования РЦ', 'код'])['Годен до'].idxmin()
]

In [16]:
data_mx.columns = const.MX_FINAL_COLUMNS

# Финальный датасет по МХ (Не стал убирать не нужный код товара, так как он был убран ранее в data_sg_mo)

In [17]:
data = data_sg_mo.merge(data_mx, how='inner', on=['Cтандартные наименования РЦ', 'Код ТП'])

In [18]:
data = data[data['Годен до на МХ'] < data['Годен до на МО']] # Основной фильтр
data['Дата отчета'] = datetime.now().date()

In [19]:
data["Дата отчета"] = pd.to_datetime(
    data["Дата отчета"],
    dayfirst=True,
    errors="coerce"
)

iso = data["Дата отчета"].dt.isocalendar()

data["Год-Неделя"] = (iso["year"].astype(str) + "-W" + iso["week"].astype(str).str.zfill(2))

In [20]:
data = data.rename(columns={
    'Cтандартные наименования РЦ':'Наименование РЦ'
})

data = data[const.FINAL_DATA_COLUMNS]


#### Сохранение в Clickhouse добавить после сохраненеия в Excel так как необходимо переименовать на анг яз колонки 

In [21]:
def save_to_excel(data: pd.DataFrame) -> None:

    date_cols = const.EXCEL_DATE_COLUMNS
    for col in date_cols:
        if col in data.columns:
            data[col] = pd.to_datetime(data[col], errors="coerce")

    # Код ТП -> число
    if "Код ТП" in data.columns:
        data["Код ТП"] = pd.to_numeric(data["Код ТП"], errors="coerce")

    report_date = pd.Timestamp("2026-03-11")


    # Добавляем округ только для внутренней логики разбивки
    rc_to_district = dict(
        zip(dim_district["RC_CODE"], dim_district["LOG_REGION"])
    )

    data_with_district = data.copy()
    data_with_district["Округ"] = data_with_district["Наименование РЦ"].map(rc_to_district)
    data_with_district["Округ"] = data_with_district["Округ"].fillna("Не распределено")

    # =========================
    # 3. Сводка TOP 20
    # =========================

    pivot = (
        data_with_district.groupby("Наименование РЦ", as_index=False)
            .size()
            .rename(columns={"size": "Количество по полю Код ТП"})
            .sort_values("Количество по полю Код ТП", ascending=False)
            .head(20)
    )

    pivot["Дата отчета"] = report_date

    # =========================
    # 4. Путь сохранения
    # =========================

    output_file = settings.OUTPUT_PATH

    # =========================
    # 5. Порядок округов и названия вкладок
    # =========================

    district_order = const.DISTRICT_ORDER

    district_sheet_names = const.DISTRICT_SHEET_NAMES

    # =========================
    # 6. Сохранение в Excel
    # =========================

    with pd.ExcelWriter(output_file, engine="xlsxwriter", datetime_format="dd.mm.yyyy") as writer:
        workbook = writer.book

        # =========================
        # Форматы
        # =========================

        title_fmt = workbook.add_format({
            "bold": True,
            "font_size": 16
        })

        text_fmt = workbook.add_format({
            "font_size": 12
        })

        header_fmt = workbook.add_format({
            "bold": True,
            "bg_color": "#1F4E78",
            "font_color": "white",
            "border": 1,
            "align": "center",
            "valign": "vcenter"
        })

        date_fmt = workbook.add_format({
            "num_format": "dd.mm.yyyy",
            "align": "center"
        })

        int_fmt = workbook.add_format({
            "num_format": "#,##0"
        })

        # =========================
        # 6.1 Лист "Сводная" - первый
        # =========================

        pivot_start_row = 14
        pivot_start_col = 3

        pivot.to_excel(
            writer,
            sheet_name="Сводная",
            index=False,
            startrow=pivot_start_row,
            startcol=pivot_start_col
        )

        ws_pivot = writer.sheets["Сводная"]

        text_col = 2       # C
        date_col = 3       # D
        right_text_col = 5 # F

        ws_pivot.write(1, text_col, "Добрый день.", text_fmt)
        ws_pivot.write(2, text_col, "Коллеги, по состоянию на", text_fmt)
        ws_pivot.write_datetime(2, date_col, report_date.to_pydatetime(), date_fmt)
        ws_pivot.write(2, right_text_col, "были выявлены ТП, пополненные не по ротации.", text_fmt)

        ws_pivot.write(4, text_col, "Это значит, что у вас на МО есть товар с более поздним СГ, чем на МХ.", text_fmt)
        ws_pivot.write(5, text_col, "Необходимо устранить выявленные отклонения и не допускать таких ошибок в дальнейшем.", text_fmt)
        ws_pivot.write(6, text_col, "Обратите внимание, что при заполнении Примечаний к паллетам необходимо учитывать корректное", text_fmt)
        ws_pivot.write(7, text_col, "написание текста в соответствии с требованиями Инструкции, избегая лишних пробелов/знаков.", text_fmt)

        ws_pivot.write(10, text_col, "HOT - на контроль и довести информацию до ответственных сотрудников.", title_fmt)
        ws_pivot.write(pivot_start_row - 2, pivot_start_col, "ТОП 20 РЦ по кол-ву пополнений не по ротации", title_fmt)

        for col_num, value in enumerate(pivot.columns):
            ws_pivot.write(pivot_start_row, pivot_start_col + col_num, value, header_fmt)

        ws_pivot.set_column("A:B", 4)
        ws_pivot.set_column("C:C", 22)
        ws_pivot.set_column("D:D", 16)
        ws_pivot.set_column("E:E", 28)
        ws_pivot.set_column("F:F", 18)
        ws_pivot.set_column("G:Z", 18)

        ws_pivot.set_row(1, 22)
        ws_pivot.set_row(2, 22)
        ws_pivot.set_row(4, 22)
        ws_pivot.set_row(5, 22)
        ws_pivot.set_row(6, 22)
        ws_pivot.set_row(7, 22)
        ws_pivot.set_row(10, 28)
        ws_pivot.set_row(pivot_start_row - 2, 28)

        qty_col_idx = pivot.columns.get_loc("Количество по полю Код ТП")
        qty_excel_col = pivot_start_col + qty_col_idx
        ws_pivot.set_column(qty_excel_col, qty_excel_col, 24, int_fmt)

        date_report_col_idx = pivot.columns.get_loc("Дата отчета")
        date_excel_col = pivot_start_col + date_report_col_idx
        ws_pivot.set_column(date_excel_col, date_excel_col, 16, date_fmt)

        first_data_row = pivot_start_row + 1
        last_data_row = pivot_start_row + len(pivot)

        ws_pivot.conditional_format(
            first_data_row, qty_excel_col,
            last_data_row, qty_excel_col,
            {
                "type": "data_bar",
                "bar_color": "#5B9BD5"
            }
        )

        # =========================
        # 6.2 Вкладки по округам - после сводной
        # =========================

        for district in district_order:
            district_df = data_with_district[data_with_district["Округ"] == district].copy()

            if district_df.empty:
                continue

            # колонку "Округ" не выгружаем
            district_df = district_df.drop(columns=["Округ"], errors="ignore")

            sheet_name = district_sheet_names[district]

            district_df.to_excel(writer, sheet_name=sheet_name, index=False)

            ws_district = writer.sheets[sheet_name]
            ws_district.autofilter(0, 0, len(district_df), len(district_df.columns) - 1)

            for idx, col in enumerate(district_df.columns):
                max_len = max(
                    len(str(col)),
                    district_df[col].astype(str).map(len).max() if not district_df.empty else 0
                )
                max_len = min(max_len + 2, 28)

                if col in date_cols:
                    ws_district.set_column(idx, idx, 15, date_fmt)
                elif col == "Код ТП":
                    ws_district.set_column(idx, idx, 14, int_fmt)
                else:
                    ws_district.set_column(idx, idx, max_len)

        # =========================
        # 6.3 Общий датасет - последний
        # =========================

        final_data_export = data_with_district.drop(columns=["Округ"], errors="ignore")

        final_data_export.to_excel(writer, sheet_name="Датасет", index=False)

        ws_data = writer.sheets["Датасет"]
        ws_data.autofilter(0, 0, len(final_data_export), len(final_data_export.columns) - 1)

        for idx, col in enumerate(final_data_export.columns):
            max_len = max(
                len(str(col)),
                final_data_export[col].astype(str).map(len).max() if not final_data_export.empty else 0
            )
            max_len = min(max_len + 2, 28)

            if col in date_cols:
                ws_data.set_column(idx, idx, 15, date_fmt)
            elif col == "Код ТП":
                ws_data.set_column(idx, idx, 14, int_fmt)
            else:
                ws_data.set_column(idx, idx, max_len)

    print(f"Файл сохранен: {output_file}")
    print(datetime.now() - start)

In [22]:
save_to_excel(data)

Файл сохранен: \\corp.tander.ru\public\Департамент качества. Отдел аналитики\03_Reporting\Отчеты\Еженедельные\Пополнение не по ротации\result.xlsx
0:08:03.891603


#### Сохраняем данные в clickhouse

In [ ]:
def prepare_df_for_clickhouse(data: pd.DataFrame) -> pd.DataFrame:

    click_df = data.copy()

    rename_map = const.CLICKHOUSE_RENAME_MAP

    click_df = click_df.rename(columns=rename_map)

    click_df['report_date'] = pd.to_datetime(click_df['report_date']).dt.date
    click_df['expiry_mo'] = pd.to_datetime(click_df['expiry_mo']).dt.date
    click_df['expiry_mx'] = pd.to_datetime(click_df['expiry_mx']).dt.date

    click_df['report_week'] = click_df['report_week'].astype(str).str.strip()
    click_df['product_code'] = click_df['product_code'].astype(str).str.strip()
    click_df['product_name'] = click_df['product_name'].astype(str).str.strip()
    click_df['rc_name'] = click_df['rc_name'].astype(str).str.strip()

    click_df['qty_mo'] = pd.to_numeric(click_df['qty_mo'], errors='coerce').fillna(0).astype(float)
    click_df['qty_mx'] = pd.to_numeric(click_df['qty_mx'], errors='coerce').fillna(0).astype(float)

    return click_df

In [ ]:
# Эта функция должна быть в pipline

def load_weekly_increment(client, click_df: pd.DataFrame) -> None:
    if click_df.empty:
        print('Датафрейм пустой. Загружать нечего.')
        return

    weeks = click_df['report_week'].dropna().unique().tolist()

    if len(weeks) != 1:
        raise ValueError(
            f'Ожидалась одна неделя в загрузке, а получено: {weeks}'
        )

    report_week = weeks[0]

    print(f'Загружаем неделю: {report_week}')

    # Удаляем старые данные за эту неделю, если они уже были загружены
    db.delete_report_week(client, report_week)

    # Загружаем новую версию
    db.insert_report_week(client, click_df)

In [25]:
click_df = prepare_df_for_clickhouse(data)
db.create_target_table(client)
load_weekly_increment(client, click_df)

Загружаем неделю: 2026-W14
Успешно загружено строк: 12832 за неделю 2026-W14
